In [1]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

#### We register our search function along with the schema from earlier lessons, without schema because docstring tells it


In [2]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [3]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"}
    )

In [4]:
agent_tools = Tools()
agent_tools.add_tool(search)
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

#### The output is the same JSON schema we hand-wrote in the function calling lesson. ToyAIKit generated it from the docstring and the type hint.



### Chat Interface

In [5]:
instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

In [9]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [15]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [16]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

# define the agent
runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [17]:
result = runner.loop(
    prompt="How do I run Olama locally?",
    callback=callback,
)

-> Response received


-> Response received


In [12]:
result.cost

CostInfo(input_cost=Decimal('0.00090225'), output_cost=Decimal('0.000918'), total_cost=Decimal('0.00182025'))

In [13]:
result.all_messages

[EasyInputMessage(content="You're a course teaching assistant.\nAnswer the QUESTION based on the CONTEXT from the FAQ database.\nUse only the facts from the CONTEXT when answering the QUESTION.", role='developer', phase=None, type=None),
 EasyInputMessage(content='How do I run Olama locally?', role='user', phase=None, type=None),
 ResponseFunctionToolCall(arguments='{"query":"Olama run locally install local setup"}', call_id='call_LUSuNfbaJjZa3rMFceMs8Y6s', name='search', type='function_call', id='fc_0cfb985505a15519006a2de8c8ec0c8195aece758314df40d4', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_LUSuNfbaJjZa3rMFceMs8Y6s',
  'output': '[\n  {\n    "id": "d09e8d4843",\n    "course": "llm-zoomcamp",\n    "section": "Module 2: Agents",\n    "question": "Install MCP Inspector",\n    "answer": "1. Ensure Node.js is installed.\\n\\n2. To install the MCP Inspector, run the following command in your terminal:\\n\\n   ```bash\\n   npm i @modelcontext

### Interactive chat

In [14]:
runner.run();

KeyboardInterrupt: Interrupted by user